<a href="https://colab.research.google.com/github/kr1stawang0510-cloud/EMSC2010_PrecipitationPirates_W7/blob/main/EMSC2010_PrecipitationPirates_W7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EMSC2010 Group Project template

## 1. Project Overview
Group name: Precipitation Pirates

Project week:7

Project title: Bayesian comparison of rainfall intensity in different ocean areas

Datasets used (name and source):

Data set 1 from `Our World Data`

“Data Page: Annual precipitation”, part of the following publication: Hannah Ritchie, Pablo Rosado, and Veronika Samborska (2024) - “Climate Change”. Data adapted from Contains modified Copernicus Climate Change Service information. Retrieved from https://archive.ourworldindata.org/20260324-093454/grapher/average-precipitation-per-year.html [online resource] (archived on March 24, 2026).

Data set 2 from `GitHub Gist`

metal3d, (2021), Info [Countries coordinates with longitude and latitude], GitHub Gist, https://gist.github.com/metal3d/5b925077e66194551df949de64e910f6

## 2. Roles and contributions

| Role | Primary | Deputy | Completed? | Notes |
| :--- | :--- | :--- | :--- | :--- |
| Github & integration | Binyao | Juliet | `Yes`/Partial/No| Inital Role complete, I will commit to GitHub as needed.|
| Data steward | Juliet | Kedi | `Yes`/Partial/No|Add note|
| Analysis  modelling | Kedi | Khai | `Yes`/Partial/No| Add note|
| Visualisation / interpretation | Khai | Lee | `Yes`/Partial/No| Included x and y axis labels for the plot. Added a caption and an interpretation of the plot.|
| Narrative | Lee | Binyao | `Yes`/Partial/No| Add note|


## 3. Deputy Interventions (if applicable)
Repeat text as required.

* Role affected:

* Reason (e.g. missed deadline, absence, etc.):

* Deputy action taken:

* Impact on workflow:

*N.B., this section should be factual, not judgemental.*

## 4. Pre-submission checklist
* Notebook runs from top to bottom.
* Datafiles are included in the GitHub repository.
* Commits include meaningful information.
* Each group member has included a brief reflection in the notebook.
* Repository has been shared with the teaching team once your project is completed.

# Start your group project here

This project aims to see how rainfall varies across different climate regions, and to explore whether equitorial regions experience higher precipitation than polar regions.

Precipitation data by country has been collected and combined with coordinate data. This allows us to classify countries based on latitide and compare the rainfall patterns across different climate regions.

In [ ]:
import pandas as pd #used to read and clean data
import matplotlib.pyplot as plt #used for data plotting

In [ ]:
#importing data
country_precip = 'average-precipitation-per-year.csv' # Document name
country_coord = 'country-coord.csv' # Document name
df_precip = pd.read_csv(country_precip) #read the data into a dataframe
df_coord = pd.read_csv(country_coord) #read the data into a dataframe

df_coord.head()

In [ ]:
#PRECIPITATION DATA
#Sorting data to only include one year (most recent)
df_precip_2025 = df_precip[df_precip['Year'] == 2025]
display(df_precip_2025.head())

In [ ]:
#sorting data to only include one year (historical)
df_precip_1950 = df_precip[df_precip['Year'] == 1950]
display(df_precip_1950.head())

In This step we first clean and organise the imported precipitation and coordinate data.

The countries are grouped based on their latitude. So equitorial regions  are between -23.43 and 23.43 degrees. while the polar regions are outside of this range.

In [ ]:
#LOCATION DATA
#filtering countries based on location
northern_hemisphere = df_coord[df_coord['Latitude (average)'] >= 0]
southern_hemisphere = df_coord[df_coord['Latitude (average)'] < 0]
equatorial_countries = df_coord[(df_coord['Latitude (average)'] >= -23.43) & (df_coord['Latitude (average)'] <= 23.43)] #between tropic of cancer and capricorn (using 'and' operator)
polar_countries = df_coord[(df_coord['Latitude (average)'] < -23.4) | (df_coord['Latitude (average)'] > 23.4)] #outside of tropics (using 'or' operator)

Next the datasets are merged so that each country is associated with both its rainfall and coordinate data.

In [ ]:
#COMBINING DATA SETS
# Merge northern hemisphere countries with their 2025 precipitation data
northern_precip_2025 = pd.merge(
    northern_hemisphere,
    df_precip_2025,
    left_on='Country',
    right_on='Entity',
    how='inner'
)

# Display the head of the combined DataFrame
display(northern_precip_2025.head())

In [ ]:
# Merge northern hemisphere countries with their 2025 precipitation data
southern_precip_2025 = pd.merge(
    southern_hemisphere,
    df_precip_2025,
    left_on='Country',
    right_on='Entity',
    how='inner'
)
# Display the head of the combined DataFrame
display(southern_precip_2025.head())

In [ ]:
#Merge equatorial countries with their 2025 precipitation data
equatorial_precip_2025 = pd.merge(
    equatorial_countries,
    df_precip_2025,
    left_on='Country',
    right_on='Entity',
    how='inner'
)
display(equatorial_precip_2025.head())

In [ ]:
# Merge polar countries with their 2025 precipitation data
polar_precip_2025 = pd.merge(
    polar_countries,
    df_precip_2025,
    left_on='Country',
    right_on='Entity',
    how='inner'
)
display(polar_precip_2025.head())

#list of data sets for analysis

For historical comparison
1. df_precip_2025
2. df_precip_1950

For locational comparsion (north/south)
1. northern_precip_2025
2. southern_precip_2025

For locational comparison (equator/poles)
1. equatorial_precip_2025
2. polar_precip_2025

In [ ]:
# Run this if you haven't installed them in Colab
# !pip install pymc arviz

import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Extracting the precipitation values for modeling
# We use the 2025 data processed by your Data Steward
data_group_a = equatorial_precip_2025['Annual precipitation'].values
data_group_b = polar_precip_2025['Annual precipitation'].values

# Summary stats for context
print(f"Equatorial Mean: {data_group_a.mean():.2f}")
print(f"Polar Mean: {data_group_b.mean():.2f}")

Comparing the percipitation between equitorial and polar suggests that equitorial regions recieve substantially more rainfall on average. with equitoral regions having a mean percipitation of 1483.97mm and polar regions with a mean percipitation of 800.15 mm. showing a 683.82 mm difference or 46% diffrence between the regions.

In [ ]:
with pm.Model() as rain_model:
    # 1. Priors for the means (Normal distributions)
    mu_a = pm.Normal("mu_a", mu=data_group_a.mean(), sigma=100)
    mu_b = pm.Normal("mu_b", mu=data_group_b.mean(), sigma=100)

    # 2. Priors for the standard deviation
    sigma_a = pm.HalfNormal("sigma_a", sigma=100)
    sigma_b = pm.HalfNormal("sigma_b", sigma=100)

    # 3. Likelihood (How well the model fits the data)
    y_a = pm.Normal("y_a", mu=mu_a, sigma=sigma_a, observed=data_group_a)
    y_b = pm.Normal("y_b", mu=mu_b, sigma=sigma_b, observed=data_group_b)

    # 4. Deterministic variable: The difference in means
    # This is what we actually want to analyze!
    diff_of_means = pm.Deterministic("difference", mu_a - mu_b)

    # 5. Sampling (Running the simulation)
    trace = pm.sample(2000, return_inferencedata=True, target_accept=0.95)

A total of 2000 samples were drawn to approximate the posterior distribution of the model parameters. The stable sampling suggests that the model provides a good fit for the data and that the estimates produced can be interpreted confidently.

In [ ]:
# Plotting the posterior distribution of the difference
az.plot_posterior(trace, var_names=["difference"], ref_val=0, hdi_prob=0.95)
plt.title("Bayesian Posterior: Difference in Rainfall Intensity")
plt.xlabel("Possible True Difference (mm)")
plt.ylabel("Posterior Density")
plt.show()

# Summary table for your project report
summary = az.summary(trace, var_names=["mu_a", "mu_b", "difference"])
print(summary)

Figure 1: Bayseian posterior distribution for the difference in mean rainfall between equatorial and polar regions. Along the x-axis are the possible true differences in rainfall in millimetres, and the y-axis shows how plausible each difference is. The 95% HDI indicates the most credible values, and the vertical orange line at x=0 represents no difference.

The posterior distribution for the difference in mean annual precipitation comparing equatorial and polar regions indicates equatorial rainfall is higher by 688mm, with a 95% HDI interval ranging from 523-836mm. Because our HDI does not include the vertical line at x=0, the plot shows strong evidence for a real difference in rainfall between the two regions. Using our knowledge of the real world, this is to be expected.

#Reflections
Binyao: I was able to set up the GitHub repository and Google Colab notebook without much difficulty. Initially, I only shared the GitHub link with my team, but they quickly pointed out that they couldn't directly save their edits in that format. Acting on this feedback, I immediately provided a shareable Colab link with editing permissions. This allowed everyone to smoothly write, run, and save their own contributions within a single, centralized document, making our collaborative coding much more efficient.


Juliet: Initally we had decided on comparing precipitation data across oceans. However, when searching I found that I did not have access to the data needed. Instead, I found yearly data for country precipitation. I thought it would be useful to group countries in a way to be compariable locationally, in additional the historical comparison. So I found an additional data set that gave countries' annual latitude and then merged it with the original data set to create specialised groups that could then be used by the analyst.

Kedi: I had some trouble with Google Colab path errors, which kept stopping my code. I fixed this by adding a simple check to make sure the CSV files were uploadwd correctly. Also, switching from basic averages to Bayesian modelling was tricky at first. I used the PyMC library to get better results. This helped the team see the actual probability of rainfall changes, making our project much more reliable.

Khai:

Lee: I was responsible for writing the explanations and ensuring the notebook had a clear structure. this role was relatively straightforward but i learnt that i needed a solid understanding of the methods that were being used inorder to translate the code into clear explanations. Also on deciding what sections should have further explanations or if the provided code comments were sufficient, so i could maintain logical flow by avoiding repetition and meaningless text.